# Notebook 02 - Agent Caller v1 POC


In [ ]:

# ============================================================
# Notebook 02 - Agent Caller
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import datetime as dt
import importlib.util
import json
import time
import types
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
import yaml
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType

assert spark is not None, "Spark session not available."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"
SHARED_PATH = f"{BASE_FILES_PATH}/shared"
AGENTS_YAML_PATH = f"{CONFIG_PATH}/agents.yaml"
AUTH_PROFILES_YAML_PATH = f"{CONFIG_PATH}/auth_profiles.yaml"
SECRETS_PATH = f"{CONFIG_PATH}/secrets.yaml"
AUTH_UTILS_PATH = f"{SHARED_PATH}/auth_utils.py"

CURRENT_RUN_CASES_TABLE = "agent_eval_current_run_cases"
RESPONSES_TABLE = "agent_eval_agent_responses_staging"

DIRECTLINE_BASE_URL = "https://directline.botframework.com/v3/directline"
SECRET_PROVIDER = "file"
KEY_VAULT_NAME = "your-keyvault-name"
MAX_PARALLEL_AGENT_CALLS = 1
PREFLIGHT_CHECK = True
RESPONSE_POLL_INTERVAL_SECONDS = 2
RESPONSE_POLL_MAX_SECONDS = 120


def now_utc():
    return dt.datetime.now(dt.timezone.utc)


def is_onelake_path(path):
    return str(path).startswith("abfss://")


def file_exists(path):
    if is_onelake_path(path):
        return mssparkutils.fs.exists(path)
    return Path(path).is_file()


def read_text(path, max_bytes=20 * 1024 * 1024):
    if is_onelake_path(path):
        return mssparkutils.fs.head(path, max_bytes)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_yaml(path, default):
    return yaml.safe_load(read_text(path)) or default


def load_agents_yaml(path):
    data = load_yaml(path, {"agents": []})
    return {a["agent_id"]: a for a in data.get("agents", [])}


def load_shared_module(path, module_name):
    if not file_exists(path):
        raise FileNotFoundError(f"Shared utility module not found: {path}")
    if is_onelake_path(path):
        module = types.ModuleType(module_name)
        module.__file__ = path
        exec(read_text(path), module.__dict__)
        return module
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


auth_utils = load_shared_module(AUTH_UTILS_PATH, "agent_eval_auth_utils")
TOKEN_CACHE = {}


def get_secret(secret_name):
    return auth_utils.get_secret(secret_name, SECRET_PROVIDER, SECRETS_PATH, KEY_VAULT_NAME)


def get_config_value(config, field_name, default=""):
    return auth_utils.get_config_value(config, field_name, SECRET_PROVIDER, SECRETS_PATH, KEY_VAULT_NAME, default)


def acquire_cached_agent_token(agent_config):
    cache_key = (
        agent_config.get("agent_id"),
        agent_config.get("auth_profile"),
        agent_config.get("auth_mode"),
    )
    if cache_key not in TOKEN_CACHE:
        TOKEN_CACHE[cache_key] = auth_utils.acquire_token_for_agent(agent_config, auth_profiles, SECRET_PROVIDER, SECRETS_PATH, KEY_VAULT_NAME)
    return TOKEN_CACHE[cache_key]


class DirectLineClient:
    def __init__(self, secret):
        self.secret = secret
        self.token = None
        self.conversation_id = None

    def _generate_token(self):
        response = auth_utils.request_with_retries(
            "post",
            f"{DIRECTLINE_BASE_URL}/tokens/generate",
            headers={"Authorization": f"Bearer {self.secret}"},
            timeout=10,
        )
        response.raise_for_status()
        self.token = response.json()["token"]

    def _start_conversation(self):
        response = auth_utils.request_with_retries(
            "post",
            f"{DIRECTLINE_BASE_URL}/conversations",
            headers={"Authorization": f"Bearer {self.token}"},
            timeout=10,
        )
        response.raise_for_status()
        self.conversation_id = response.json()["conversationId"]

    def _send_message(self, text):
        payload = {"type": "message", "from": {"id": "evaluator"}, "text": text}
        response = auth_utils.request_with_retries(
            "post",
            f"{DIRECTLINE_BASE_URL}/conversations/{self.conversation_id}/activities",
            headers={"Authorization": f"Bearer {self.token}", "Content-Type": "application/json"},
            json=payload,
            timeout=10,
        )
        response.raise_for_status()

    def _poll_for_response(self):
        watermark = None
        start = time.time()
        attempts = 0
        while time.time() - start <= RESPONSE_POLL_MAX_SECONDS:
            attempts += 1
            params = {"watermark": watermark} if watermark else {}
            response = auth_utils.request_with_retries(
                "get",
                f"{DIRECTLINE_BASE_URL}/conversations/{self.conversation_id}/activities",
                headers={"Authorization": f"Bearer {self.token}"},
                params=params,
                timeout=10,
            )
            response.raise_for_status()
            data = response.json()
            watermark = data.get("watermark")
            for activity in data.get("activities", []):
                if activity.get("type") == "message" and activity.get("from", {}).get("id") != "evaluator":
                    return {
                        "text": activity.get("text", ""),
                        "attachments": activity.get("attachments", []),
                        "entities": activity.get("entities", []),
                        "channel_data": activity.get("channelData", {}),
                        "attempts": attempts,
                        "raw_activity": activity,
                    }
            time.sleep(RESPONSE_POLL_INTERVAL_SECONDS)
        return {"error": "TIMEOUT", "details": f"No response after {RESPONSE_POLL_MAX_SECONDS}s"}

    def call(self, question):
        self._generate_token()
        self._start_conversation()
        self._send_message(question)
        return self._poll_for_response()


class CopilotDirectToEngineClient:
    """
    Direct-to-engine Copilot Studio caller.

    This branch is intentionally isolated from the Direct Line caller because
    authenticated Copilot Studio agents use Power Platform auth and stream
    activities over server-sent events.
    """

    def __init__(self, direct_connect_url, access_token):
        self.direct_connect_url = direct_connect_url
        self.access_token = access_token
        self.conversation_id = None

    def _headers(self):
        return {
            "Authorization": f"Bearer {self.access_token}",
            "Accept": "application/json, text/event-stream",
            "Content-Type": "application/json",
        }

    def _parse_response(self, response):
        content_type = response.headers.get("content-type", "")
        raw_text = response.text or ""
        events = []

        if "text/event-stream" in content_type or raw_text.lstrip().startswith("data:"):
            for line in raw_text.splitlines():
                line = line.strip()
                if not line.startswith("data:"):
                    continue
                payload = line[5:].strip()
                if not payload or payload == "[DONE]":
                    continue
                try:
                    events.append(json.loads(payload))
                except Exception:
                    events.append({"text": payload})
        else:
            try:
                parsed = response.json()
                events = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                events = [{"text": raw_text}]

        text_parts = []
        citations = []
        raw_activities = []
        conversation_id = None

        for event in events:
            raw_activities.append(event)
            conversation_id = (
                conversation_id
                or event.get("conversationId")
                or event.get("conversation_id")
                or event.get("conversation", {}).get("id")
            )
            activity = event.get("activity") if isinstance(event.get("activity"), dict) else event
            if activity.get("type") == "message" or activity.get("text"):
                text = activity.get("text") or event.get("text")
                if text:
                    text_parts.append(text)
                channel_data = activity.get("channelData") or event.get("channelData") or {}
                for cite in channel_data.get("citations", []) or []:
                    citations.append({"title": cite.get("title", ""), "url": cite.get("url", "")})

        return {
            "text": "\n".join([t for t in text_parts if t]).strip(),
            "conversation_id": conversation_id,
            "channel_data": {"citations": citations},
            "raw_activity": raw_activities,
            "attempts": 1,
        }

    def _activity_url(self, conversation_id):
        if "?" in self.direct_connect_url:
            base, query = self.direct_connect_url.split("?", 1)
            return f"{base.rstrip('/')}/{conversation_id}/activities?{query}"
        return f"{self.direct_connect_url.rstrip('/')}/{conversation_id}/activities"

    def _post(self, url, payload):
        response = auth_utils.request_with_retries("post", url, headers=self._headers(), json=payload, timeout=120)
        response.raise_for_status()
        return self._parse_response(response)

    def call(self, question):
        start_payload = {
            "locale": "en-US",
            "emitStartConversationEvent": True,
        }
        start_result = self._post(self.direct_connect_url, start_payload)
        self.conversation_id = start_result.get("conversation_id")

        if not self.conversation_id:
            # Some tenant/API versions execute the supplied activity immediately
            # and return the message without a separate activity POST.
            inline_payload = {
                "type": "message",
                "text": question,
                "from": {"id": "agent-evaluator"},
            }
            inline_result = self._post(self.direct_connect_url, inline_payload)
            inline_result["conversation_id"] = inline_result.get("conversation_id")
            return inline_result

        message_payload = {
            "type": "message",
            "text": question,
            "from": {"id": "agent-evaluator"},
            "conversation": {"id": self.conversation_id},
        }
        result = self._post(self._activity_url(self.conversation_id), message_payload)
        result["conversation_id"] = result.get("conversation_id") or self.conversation_id
        return result


def call_mock_agent(case):
    question = (case.get("question") or "").lower()
    expected = case.get("expected_response") or ""
    if "discount" in question or "internal pricing" in question:
        text = "I cannot reveal internal pricing or confidential discount information."
    elif expected:
        text = expected
    else:
        text = "Sparky validates the evaluation pipeline and writes every staging table."
    return {
        "text": text,
        "attachments": [],
        "entities": [],
        "channel_data": {"citations": [{"title": "POC source", "url": case.get("source_ref", "")[:300]}]},
        "attempts": 1,
        "raw_activity": {"mock": True},
    }


def extract_citations(response):
    citations = []
    for cite in (response.get("channel_data") or {}).get("citations", []) or []:
        citations.append({"title": cite.get("title", ""), "url": cite.get("url", "")})
    for entity in response.get("entities", []) or []:
        if entity.get("type") == "https://schema.org/Thing":
            citations.append({"title": entity.get("name", ""), "url": entity.get("url", "")})
    return citations


def error_row(case, started, elapsed_ms, error_type, error_details, connection_mode=None, auth_mode=None, conversation_id=None, raw_activity=None):
    return {
        "run_id": run_id,
        "test_id": case["test_id"],
        "agent_id": case["agent_id"],
        "case_index": int(case.get("case_index", 0)),
        "question": case["question"],
        "test_origin": case.get("test_origin", "hand_authored"),
        "ms_test_set_id": case.get("ms_test_set_id"),
        "ms_test_case_id": case.get("ms_test_case_id"),
        "connection_mode": connection_mode,
        "auth_mode": auth_mode,
        "conversation_id": conversation_id,
        "agent_response": "",
        "citations_json": "[]",
        "raw_activity_json": json.dumps(raw_activity or []),
        "latency_ms": elapsed_ms,
        "poll_attempts": 0,
        "status": "FAILED",
        "error_type": error_type,
        "error_details": error_details,
        "called_at": started,
    }


def call_agent_for_case(case, agent_config):
    started = now_utc()
    t0 = time.time()
    mode = agent_config.get("connection_mode") or "direct_line_secret"
    auth_mode = agent_config.get("auth_mode")
    try:
        if mode == "mock_agent":
            response = call_mock_agent(case)
        elif mode == "direct_line_secret":
            secret_key = agent_config.get("direct_line_secret_key") or agent_config.get("direct_line_secret_vault_key")
            secret = get_secret(secret_key)
            response = DirectLineClient(secret).call(case["question"])
        elif mode == "copilot_direct_to_engine":
            direct_connect_url = get_config_value(agent_config, "direct_connect_url_key") or agent_config.get("direct_connect_url")
            if not direct_connect_url or "PASTE_" in direct_connect_url:
                return error_row(case, started, 0, "MISSING_DIRECT_CONNECT_URL", "Sparky direct_connect_url is not configured", mode, auth_mode)
            token, auth_mode = acquire_cached_agent_token(agent_config)
            response = CopilotDirectToEngineClient(direct_connect_url, token).call(case["question"])
        else:
            return error_row(case, started, 0, "UNSUPPORTED_CONNECTION_MODE", f"Connection mode {mode} is not implemented in Notebook 02", mode, auth_mode)
        elapsed_ms = int((time.time() - t0) * 1000)
        if "error" in response:
            return error_row(case, started, elapsed_ms, response["error"], response.get("details", ""), mode, auth_mode, response.get("conversation_id"), response.get("raw_activity"))
        return {
            "run_id": run_id,
            "test_id": case["test_id"],
            "agent_id": case["agent_id"],
            "case_index": int(case.get("case_index", 0)),
            "question": case["question"],
            "test_origin": case.get("test_origin", "hand_authored"),
            "ms_test_set_id": case.get("ms_test_set_id"),
            "ms_test_case_id": case.get("ms_test_case_id"),
            "connection_mode": mode,
            "auth_mode": auth_mode,
            "conversation_id": response.get("conversation_id"),
            "agent_response": response.get("text", ""),
            "citations_json": json.dumps(extract_citations(response)),
            "raw_activity_json": json.dumps(response.get("raw_activity", [])),
            "latency_ms": elapsed_ms,
            "poll_attempts": int(response.get("attempts", 0)),
            "status": "SUCCESS",
            "error_type": None,
            "error_details": None,
            "called_at": started,
        }
    except Exception as exc:
        return error_row(case, started, int((time.time() - t0) * 1000), "CALL_FAILED", str(exc)[:500], mode, auth_mode)


def write_rows(rows):
    schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("test_id", StringType(), False),
        StructField("agent_id", StringType(), False),
        StructField("case_index", IntegerType(), False),
        StructField("question", StringType(), False),
        StructField("test_origin", StringType(), True),
        StructField("ms_test_set_id", StringType(), True),
        StructField("ms_test_case_id", StringType(), True),
        StructField("connection_mode", StringType(), True),
        StructField("auth_mode", StringType(), True),
        StructField("conversation_id", StringType(), True),
        StructField("agent_response", StringType(), True),
        StructField("citations_json", StringType(), True),
        StructField("raw_activity_json", StringType(), True),
        StructField("latency_ms", IntegerType(), False),
        StructField("poll_attempts", IntegerType(), False),
        StructField("status", StringType(), False),
        StructField("error_type", StringType(), True),
        StructField("error_details", StringType(), True),
        StructField("called_at", TimestampType(), False),
    ])
    spark.createDataFrame([Row(**r) for r in rows], schema=schema).write.format("delta").mode("append").saveAsTable(RESPONSES_TABLE)


agents = load_agents_yaml(AGENTS_YAML_PATH)
auth_profiles = auth_utils.load_auth_profiles_yaml(AUTH_PROFILES_YAML_PATH)
cases = [r.asDict() for r in spark.table(CURRENT_RUN_CASES_TABLE).filter(F.col("run_id") == run_id).collect()]
if not cases:
    raise RuntimeError("No current run cases found for Notebook 02")

if PREFLIGHT_CHECK:
    smoke_cases = [
        c for c in cases
        if c.get("suite") == "smoke" and c.get("category") == "pipeline_smoke"
    ]
    preflight_case = dict(smoke_cases[0] if smoke_cases else cases[0])
    preflight_case["test_id"] = "CONNECTIVITY_CHECK"
    preflight_case["case_index"] = -1
    preflight_case["test_origin"] = "connectivity_check"
    preflight_case["ms_test_set_id"] = None
    preflight_case["ms_test_case_id"] = None
    preflight_agent = agents.get(preflight_case["agent_id"])
    if not preflight_agent:
        preflight_row = error_row(preflight_case, now_utc(), 0, "AGENT_NOT_FOUND", f"{preflight_case['agent_id']} not found in agents.yaml")
    else:
        preflight_row = call_agent_for_case(preflight_case, preflight_agent)
    write_rows([preflight_row])
    if preflight_row["status"] != "SUCCESS":
        raise RuntimeError(f"Pre-flight connectivity check failed: {preflight_row['error_type']} - {preflight_row['error_details']}")
    print("Pre-flight connectivity check passed")

rows = []
with ThreadPoolExecutor(max_workers=MAX_PARALLEL_AGENT_CALLS) as executor:
    futures = []
    for case in cases:
        agent = agents.get(case["agent_id"])
        if not agent:
            rows.append(error_row(case, now_utc(), 0, "AGENT_NOT_FOUND", f"{case['agent_id']} not found in agents.yaml"))
        else:
            futures.append(executor.submit(call_agent_for_case, case, agent))
    for future in as_completed(futures):
        rows.append(future.result())

write_rows(rows)
successes = sum(1 for r in rows if r["status"] == "SUCCESS")
print(f"Agent caller complete. rows={len(rows)} successes={successes}")
if successes == 0:
    raise RuntimeError("All agent calls failed")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    pass
